In [93]:
import pandas as pd
import re
import json
from IPython.display import FileLink, display, HTML

In [94]:
#1. load data
finentity_dataset = pd.read_json("multilingual_finentity.json")

In [95]:
print(finentity_dataset.head())

  language                                               text  \
0   telugu  ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...   
1   telugu  లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...   
2   telugu  హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించి...   
3   telugu  టాటా స్టీక్సా నిలకడగా వృద్ధి చూపించింది, మధ్యస...   
4    hindi  Reliance ने इस तिमाही में 10% की आमदनी वृद्धि ...   

                                            entities  
0  [{'start': 0, 'end': 8, 'value': 'ఇన్ఫొసిస్', ...  
1  [{'start': 0, 'end': 18, 'value': 'లార్సెన్ & ...  
2  [{'start': 0, 'end': 16, 'value': 'హిందుస్థాన్...  
3  [{'start': 0, 'end': 9, 'value': 'టాటా స్టీ', ...  
4  [{'start': 0, 'end': 8, 'value': 'Reliance', '...  


In [96]:
sls_data = finentity_dataset[:-1]["text"]

In [97]:
print(sls_data[:10])

0    ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...
1    లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...
2    హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించి...
3    టాటా స్టీక్సా నిలకడగా వృద్ధి చూపించింది, మధ్యస...
4    Reliance ने इस तिमाही में 10% की आमदनी वृद्धि ...
5    Tata Motors को वाहन संकट और लागत दबाव के चलते ...
6    HDFC बैंक की ऋण दरों में स्थिरता बनी रही — बाज...
7    Infosys ने ठोस तकनीकी समझौते हासिल किए, जिससे ...
8    Banco Santander reportó pérdidas inesperadas e...
9    Iberdrola aumentó su dividend yield y sorprend...
Name: text, dtype: object


In [98]:
weighted_words = {
    # 🟢 Positive / Profit / Uptrend (Expanded with more financial-specific terms and balanced weights)
    "increase": 1, "rise": 1, "gain": 1, "growth": 1, "improve": 1, "advance": 1,
    "up": 1, "higher": 1, "recover": 2, "stabilize": 1, "bounce": 1,
    "surge": 3, "boost": 2, "record high": 5, "upward": 2, "expand": 2, 
    "jump": 3, "climb": 2, "rally": 3, "appreciate": 2,
    "profit": 3, "strong performance": 4, "beat expectations": 4,
    "outperform": 3, "bull run": 4, "gain momentum": 2, "strengthen": 2,
    "growth momentum": 3, "high demand": 3, "positive returns": 3,
    "above forecast": 3, "solid results": 3, "better than expected": 4,
    "turnaround": 3, "exceed targets": 4, "record profits": 5,
    "margin expansion": 4, "earnings beat": 4, "strong cash flow": 3,
    "return to growth": 3, "upside potential": 3, "robust performance": 4,
    "record revenue": 5, "strong earnings report": 4, "profit margin improvement": 4,
    "market share growth": 3, "revenue growth": 3, "positive guidance": 4,
    "higher margins": 4, "strong bookings": 3, "dividend increase": 3,
    "acquisition": 2, "merger": 2, "inflow": 2, "net income": 3, "share buyback": 3,
    # New additions for financial depth
    "revenue beat": 4, "strong balance sheet": 4, "cost savings": 3, "efficiency gains": 3,
    "synergies": 3, "strategic acquisition": 3, "market leadership": 3, "brand strength": 3,
    "innovation": 2, "r&d success": 3, "product launch": 2, "customer growth": 3,
    "subscription increase": 3, "order backlog": 2, "sales pipeline": 2, "asset appreciation": 2,
    "eps growth": 3, "free cash flow": 3, "roic improvement": 3,
    
    # 🟩 Positive Sentiment / Guidance / Outlook (Enhanced forward-looking terms with higher weights for specificity)
    "bullish": 3, "positive outlook": 4, "raised guidance": 4,
    "upgrade": 3, "high return": 3, "good earnings": 2, "recovery": 3,
    "optimistic": 2, "confidence": 2, "buy recommendation": 4,
    "outlook improved": 4, "growth prospects": 3, "expansion": 3,
    "record quarter": 4, "resilient": 2, "beat estimates": 4,
    "strong fundamentals": 4, "positive momentum": 3, "favorable trends": 3,
    "accelerating": 2, "promising": 2, "well-positioned": 3, "favorable": 2,
    # New additions
    "upgraded forecast": 4, "positive revision": 4, "strong pipeline": 3,
    "favorable conditions": 3, "tailwinds": 3, "catalyst": 3, "milestone": 2,
    "breakthrough": 3, "strategic partnership": 2, "alliance": 2, "analyst upgrade": 3,
    "consensus beat": 4, "guidance raise": 4, "long-term growth": 3,
    
    # 🔴 Negative / Loss / Decline (Stronger penalties and more event-specific terms)
    "decrease": -1, "drop": -1, "fall": -1, "loss": -4, "decline": -2,
    "reduce": -1, "dip": -1, "down": -1, "lower": -1, "slight drop": -1,
    "slump": -3, "weaken": -2, "negative growth": -3, "cut": -2, "layoff": -4,
    "missed estimates": -4, "bankruptcy": -6, "crash": -5, "collapse": -5,
    "bear market": -5, "selloff": -4, "downgrade": -4, "bearish": -4,
    "warning": -3, "losses widened": -5, "profit warning": -4,
    "underperform": -3, "negative results": -4, "below expectations": -4,
    "falling demand": -3, "sluggish": -2, "poor performance": -3,
    "below forecast": -3, "drop sharply": -5, "plunge": -5, "tumble": -5,
    "earnings miss": -4, "decline in revenue": -4, "operational challenges": -3,
    "cost overrun": -3, "write-down": -5, "asset impairment": -5, "underwhelming": -3,
    "weak sales": -4, "profit decline": -4, "missed guidance": -4,
    "debt burden": -3, "liquidation": -5, "defaults": -5, "outflow": -2,
    # New additions
    "impairment charge": -5, "restructuring costs": -3, "lawsuit": -4,
    "regulatory fine": -4, "sanction": -4, "fraud allegation": -5, "product recall": -4,
    "product defect": -4, "cyber attack": -4, "data breach": -4, "inventory writedown": -4,
    "goodwill impairment": -5, "margin contraction": -4, "cash burn": -3,
    "dividend cut": -4, "credit downgrade": -4, "insider selling": -3,
    
    # 🟠 Risk / Volatility / Uncertainty (Expanded with modern financial risks, weighted for persistence)
    "downturn": -3, "slowdown": -3, "volatile": -2, "risk": -2, "inflation": -3,
    "inflation concerns": -4, "negative outlook": -4, "deficit": -3,
    "uncertainty": -3, "market correction": -4, "economic crisis": -5,
    "recession": -5, "credit risk": -4, "liquidity issues": -4,
    "macroeconomic headwinds": -4, "tightening": -3, "concerns": -3,
    "declining margins": -4, "supply chain issues": -3, "volatility": -2,
    "geopolitical risk": -4, "policy uncertainty": -4, "regulatory risk": -4,
    "currency fluctuation": -2, "interest rate hike": -3, "market turbulence": -4,
    "downgrade outlook": -4, "potential default": -5, "austerity": -3,
    # New additions
    "trade tension": -3, "tariff impact": -4, "supply disruption": -3,
    "labor shortage": -3, "currency risk": -3, "commodity volatility": -3,
    "climate risk": -3, "esg concerns": -3, "competition intensifies": -3,
    "rate cut delay": -2, "fiscal cliff": -4, "sovereign debt": -4,
    
    # 🔵 Neutral / Contextual / Low Sentiment (Expanded for better financial context handling in sequences)
    "announces": 0, "reports": 0, "analyst": 0, "forecast": 0, "expects": 0,
    "market": 0, "stock": 0, "shares": 0, "investors": 0, "trading": 0,
    "held": 0, "remains": 0, "unchanged": 0, "flat": 0, "steady": 0,
    "according to": 0, "says": 0, "issued": 0, "released": 0, "data": 0,
    "securities": 0, "brokerage": 0, "management": 0,
    # New additions
    "quarter": 0, "year": 0, "fiscal": 0, "gaap": 0, "eps": 0,
    "p/e": 0, "dividend": 0, "yield": 0, "volume": 0, "price": 0,
    "index": 0, "sector": 0, "industry": 0, "portfolio": 0, "fund": 0,
    "preliminary": 0, "final": 0, "consensus": 0, "estimated": 0, "projected": 0,
    
    # ⚡ Context modifiers (Enhanced for sequence-level nuance, including more negations and financial qualifiers)
    "slightly": 0.5, "marginally": 0.5, "modestly": 0.5,
    "significantly": 1.5, "sharply": 1.5, "substantially": 1.5, "dramatically": 1.75,
    "not": -1, "barely": 0.5, "still": 1,
    "exceptionally": 2, "severely": 2, "record": 1.5,
    "despite": -0.5,
    "however": 0, "but": 0,
    # New additions
    "no": -1, "never": -1, "none": -1, "without": -0.5,
    "expected": 0.5, "anticipated": 0.5, "major": 1.5, "key": 1.2,
    "ongoing": 0, "initial": 0, "subsequent": 0, "cumulative": 0,
    "versus": 0, "compared to": 0, "relative to": 0
}

In [99]:
sentence_sentiment_dataset = []

for sentence in sls_data:
    words = [w for w in re.split(r'[\/, .?!*&^%$#@=+]+', sentence) if w]
    score =0
    for word in words:
        word = word.lower()
        if(word in weighted_words ):
            score += weighted_words[word]
    # if score above certain threshold then positive
    if(score >= 1 ): 
        sentiment ="Positive"
    elif(score <= -1):
        sentiment = "Negative" # if score below certain threshold then negative
    else:
        sentiment = "Neutral"
    sentence_sentiment_dataset.append({
        "sentence" : sentence, 
        "sentiment" : sentiment
    })

In [100]:
neutral_count =0
positive_count = 0
negative_count =0
for data in sentence_sentiment_dataset:
    print(f"sentence:  {data['sentence']}\nsentiment: {data['sentiment']}")
    print()
    if(data['sentiment']=="Neutral"):
        neutral_count+=1
    elif(data['sentiment']=="Positive"):
        positive_count +=1
    else:
        negative_count +=1
        

sentence:  ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూపించింది, ఆపు పాజిటివ్ స్వరాలు తగ్గాయి.
sentiment: Neutral

sentence:  లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్ర‌మంలో పెరిగాయి, షేర్లు 5%కి పైగా బలపడ్డాయి.
sentiment: Neutral

sentence:  హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించింది — కంపెనీ పరిమిత ఆదాయాలతో బాధపడుతోంది.
sentiment: Neutral

sentence:  టాటా స్టీక్సా నిలకడగా వృద్ధి చూపించింది, మధ్యస్థితిలో న్యూట్రల్ మార్కెట్ టోన్.
sentiment: Neutral

sentence:  Reliance ने इस तिमाही में 10% की आमदनी वृद्धि दर्ज की, निवेशकों में उत्साह बढ़ा।
sentiment: Neutral

sentence:  Tata Motors को वाहन संकट और लागत दबाव के चलते नुकसान का सामना करना पड़ा।
sentiment: Neutral

sentence:  HDFC बैंक की ऋण दरों में स्थिरता बनी रही — बाजार में तटस्थ प्रतिक्रिया रही।
sentiment: Neutral

sentence:  Infosys ने ठोस तकनीकी समझौते हासिल किए, जिससे विश्वास में सुधार हुआ।
sentiment: Neutral

sentence:  Banco Santander reportó pérdidas inesperadas este trimestre debido a riesgos macroeconómicos.
sent

In [101]:
print(f"neutral: {neutral_count}")
print(f"positive: {positive_count}")
print(f"negative: {negative_count}")
print(f"total sentences: {len(sentence_sentiment_dataset)}")

neutral: 69
positive: 1
negative: 0
total sentences: 70


In [122]:
# Convert to new format
new_data = []

for item in sentence_sentiment_dataset:
    new_item = {
        "sentence": item["sentence"],
        "entity": "none",  # sentence-level sentiment → entity is none
        "sentiment": item["sentiment"]
    }
    new_data.append(new_item)
sentence_sentiment_dataset=new_data

In [123]:
with open("sentenceSentimentDataset.json", "w", encoding="utf-8") as json_file:
    json.dump(sentence_sentiment_dataset, json_file, ensure_ascii=False, indent=4)  # indent=4 for pretty formatting
FileLink("sentenceSentimentDataset.json")

C:\Users\JAYASHREE\Downloads\EntityPulse\Models\sentenceSentimentDataset.json

In [124]:
display(HTML('<a href="sentenceSentimentDataset.json" download>Click here to download JSON</a>'))

In [125]:
sls_data = pd.DataFrame(output)
sls_data.to_csv("Sentences.csv", index=False, encoding='utf-8-sig')
display(HTML("""
<a href="Sentences.csv" download="Sentences.csv" target="_blank">
Click here to download CSV
</a>
"""))

In [126]:
sentence_sentiment_data = pd.read_json("sentenceSentimentDataset.json")
sentence_entity_combined_sentiment_dataset=[]
sentence_index = 0

data = []
for row, sent_row in zip(finentity_dataset.itertuples(), sentence_sentiment_data.itertuples()):
    sentence = row.text
    sentence_entity_combined_sentiment_dataset.append([sentence, "none", sent_row.sentiment])
    for en in row.entities:
        entity = en.get('value')
        sentiment = en.get('tag')
        sentence_entity_combined_sentiment_dataset.append([sentence, entity, sentiment])

df_flat = pd.DataFrame(sentence_entity_combined_sentiment_dataset, columns=['sentence', 'entity', 'sentiment'])
print("\nFlattened dataset sample:\n", df_flat.head(20))


Flattened dataset sample:
                                              sentence                 entity  \
0   ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...                   none   
1   ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...              ఇన్ఫొసిస్   
2   లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...                   none   
3   లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...      లార్సెన్ & టూర్బో   
4   హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించి...                   none   
5   హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించి...  హిందుస్థాన్ యూనిలివర్   
6   టాటా స్టీక్సా నిలకడగా వృద్ధి చూపించింది, మధ్యస...                   none   
7   టాటా స్టీక్సా నిలకడగా వృద్ధి చూపించింది, మధ్యస...              టాటా స్టీ   
8   Reliance ने इस तिमाही में 10% की आमदनी वृद्धि ...                   none   
9   Reliance ने इस तिमाही में 10% की आमदनी वृद्धि ...               Reliance   
10  Tata Motors को वाहन संकट और लागत दबाव के चलते ...                   none   
11  Tata Mot

In [127]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[A-Z]+\.[A-Z]+>', '', text)
    text = re.sub(r'[^A-Za-z0-9.,!?;\'\"\s&]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nBefore cleaning:\n", df_flat['sentence'].head(5))
df_flat['sentence'] = df_flat['sentence'].apply(clean_text)
print("\nAfter cleaning:\n", df_flat['sentence'].head(5))


Before cleaning:
 0    ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...
1    ఇన్ఫొసిస్ ఈ త్రైమాసికంలో ఆదాయంలో 8% తగ్గుదల చూ...
2    లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...
3    లార్సెన్ & టూర్బో ఫార్మా భాగస్వామ్యాలు భారీ క్...
4    హిందుస్థాన్ యూనిలివర్ నికర నష్టాన్ని ప్రకటించి...
Name: sentence, dtype: object

After cleaning:
 0      8 , .
1      8 , .
2    & , 5 .
3    & , 5 .
4          .
Name: sentence, dtype: object


In [128]:
df_flat.to_csv("SentenceEntityCombinedSentimentDataset.csv", index =False)